# Proyecto final: clasificación de señales EOG con MLP

Se estudian diez clases del lenguaje Blink-to-Speak.

**Objetivo:** comparar una MLP con las 29 características permitidas, sin `Longitud`, y otra con características reducidas por correlación.

La implementación se encuentra en `proyecto_mlp.py`. Las dependencias están en `requirements.txt`.


In [ ]:
from pathlib import Path
import sys

base = Path.cwd()
if not (base / "proyecto_mlp.py").exists():
    base = next((p for p in [base / "code", base / "proyecto-final" / "code"]
                 if (p / "proyecto_mlp.py").exists()), base)
sys.path.insert(0, str(base))
import pandas as pd
from IPython.display import display, Image, Markdown
from proyecto_mlp import cargar_datos, ejecutar

dataset = base.parent / "DataSet_completo.xlsx"
salida = base / "resultados"


## 1. Datos y preprocesamiento

El archivo contiene 200 registros, 20 por clase, con características numéricas ya calculadas. No se vuelven a filtrar señales crudas porque no están disponibles. Se excluyen `Longitud` y el objetivo `Clase` de los predictores. La ejecución verifica clases, faltantes, infinitos y constantes.

El orden de filas es por clase; usamos particiones aleatorias estratificadas: 60% entrenamiento, 20% validación y 20% prueba, semilla 42. Cada StandardScaler se ajusta solo con entrenamiento. Las filas se conservan para comparar ambas fases con las mismas observaciones.


In [ ]:
datos, X, y = cargar_datos(dataset)
display(pd.DataFrame({"filas": [len(datos)], "predictores_sin_longitud": [X.shape[1]],
                      "faltantes": [datos.isna().sum().sum()], "duplicados": [datos.duplicated().sum()]}))
display(y.value_counts().sort_index().rename("registros_por_clase").to_frame())
display(X.head())


## 2–4. MLP original, correlación y MLP reducido

Se prueban tres configuraciones pequeñas: `(16,)` ReLU, `(32,)` tanh y `(32,16)` ReLU. Adam, regularización alpha=0.001, learning rate=0.001 y lote de 16 son comunes. Se observan pérdida y accuracy de entrenamiento/validación durante 600 épocas.

Cada configuración original selecciona época por mayor accuracy de validación; el desempate usa menor pérdida de validación. La mejor configuración global se selecciona del mismo modo en esa fase. El conjunto de prueba no participa en estas decisiones.

La matriz de Pearson se calcula solo en entrenamiento. Si **|r| > 0.8**, se elimina una columna posterior redundante con una ya conservada según el orden del Excel. El resultado conserva `Clase` y unidades originales. La MLP reducida repite las tres configuraciones, y usa exactamente la misma época de su pareja original; se generan sus curvas también. Solo la pareja elegida accede a prueba.

La siguiente celda realiza todo el experimento y guarda evidencias en `resultados/`. Volver a ejecutarla reemplaza esos resultados con la misma semilla.


In [ ]:
comparacion_validacion, metricas_prueba = ejecutar(dataset, salida, epocas=600)
display(comparacion_validacion)
display(metricas_prueba)


## Selección y visualizaciones

Las líneas verticales de las curvas señalan la época realmente usada. La curva de validación permite identificar si continuar entrenando mejora o solo reduce la pérdida de entrenamiento. No se elige retrospectivamente por el resultado de prueba.


In [ ]:
display(pd.read_csv(salida / "caracteristicas_eliminadas.csv"))
display(pd.read_csv(salida / "caracteristicas_conservadas.csv"))
display(Image(filename=str(salida / "correlacion_entrenamiento.png")))
for figura in sorted(salida.glob("curvas_*.png")):
    display(Image(filename=str(figura)))


## Evaluación final en prueba

Se reportan accuracy, precision, recall y F1 macro; las matrices y reportes por clase muestran qué patrones se confunden. Hay 40 registros de prueba, de modo que un acierto cambia accuracy en 2.5 puntos porcentuales. Una sola partición no establece una mejora generalizable.


In [ ]:
display(Image(filename=str(salida / "comparacion_prueba.png")))
for fase in ["sin_longitud", "reducido"]:
    display(Image(filename=str(salida / f"matriz_confusion_{fase}.png")))
    display(pd.read_csv(salida / f"reporte_clases_{fase}.csv", index_col=0))


## 5. Análisis de resultados

El artículo reporta **83.4%** con DT de MATLAB, longitud incluida, 19 características, cinco folds y diez entrenamientos (pp. 5–7). Ese resultado se usa como referencia: cambian variables, algoritmo y validación. El artículo no enumera exactamente qué columnas eliminó por correlación; que el árbol no use suma/producto no prueba su eliminación previa.

No hay identificador de participante para evaluar generalización a personas nuevas. El balance y la ausencia de faltantes no compensan por sí solos el tamaño de 200 registros ni la adquisición controlada.

A continuación se presenta el análisis de las métricas y las preguntas del proyecto.


In [ ]:
display(Markdown((salida / "notas_para_reporte.md").read_text(encoding="utf-8")))
